In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime



In [2]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

print("Processed folder ready:", processed_dir)

Processed folder ready: ..\data\processed


In [3]:
fact = pd.read_parquet(
    "../data/interim/fact_validations.parquet"
)

print("Fact table shape:", fact.shape)

Fact table shape: (79149, 34)


In [4]:
fact = fact.rename(columns={
    "district_x": "route_district",
    "district_y": "stop_district"
})

print("District columns renamed successfully:")
print([col for col in fact.columns if "district" in col])

District columns renamed successfully:
['route_district', 'stop_district']


In [5]:
print("FACT COLUMNS:")
print(fact.columns.tolist())

print("\nFACT SHAPE:")
print(fact.shape)


FACT COLUMNS:
['validation_id', 'validation_ts', 'route_id', 'vehicle_id', 'stop_id', 'card_id', 'fare_type', 'fare_amount', 'passenger_count', 'transfer_flag', 'is_reversed', 'date', 'year', 'month', 'year_month', 'quarter', 'hour', 'day_of_week', 'is_weekend', 'time_period', 'revenue', 'route_name', 'mode', 'route_district', 'route_length_km', 'vehicle_type', 'depot', 'vehicle_age_band', 'capacity', 'stop_name', 'stop_district', 'zone', 'has_shelter', 'load_factor']

FACT SHAPE:
(79149, 34)


In [6]:
fact["day_type"] = fact["is_weekend"].map({
    False: "Weekday",
    True: "Weekend"
})

print(fact[["hour", "is_weekend", "day_type"]].head())

   hour  is_weekend day_type
0    18       False  Weekday
1     8       False  Weekday
2    13       False  Weekday
3    17        True  Weekend
4    18       False  Weekday


In [7]:
agg_ridership_by_month = (
    fact.groupby("year_month")
    .agg(
        total_boardings=("passenger_count", "sum"),
        total_revenue=("revenue", "sum"),
        unique_cards=("card_id", "nunique"),
        avg_daily_boardings=("passenger_count", "mean")
    )
    .reset_index()
)

agg_ridership_by_month["total_revenue"] = (
    agg_ridership_by_month["total_revenue"].round(2)
)

agg_ridership_by_month["avg_daily_boardings"] = (
    agg_ridership_by_month["avg_daily_boardings"].round(4)
)

agg_ridership_by_month = agg_ridership_by_month.sort_values(
    "year_month"
)

print(agg_ridership_by_month)

  year_month  total_boardings  total_revenue  unique_cards  \
0    2024-01            14581       26798.67          9446   
1    2024-02            13607       25176.67          9039   
2    2024-03            14531       26589.69          9428   
3    2024-04            13933       25620.73          9257   
4    2024-05            14671       26714.24          9512   
5    2024-06            13924       25590.24          9154   

   avg_daily_boardings  
0               1.0753  
1               1.0793  
2               1.0795  
3               1.0723  
4               1.0765  
5               1.0795  


In [8]:

daily_boardings = (
    fact.groupby(["year_month", "date"])
    .agg(
        daily_boardings=("passenger_count", "sum")
    )
    .reset_index()
)

avg_daily = (
    daily_boardings.groupby("year_month")
    .agg(
        avg_daily_boardings=("daily_boardings", "mean")
    )
    .reset_index()
)

avg_daily["avg_daily_boardings"] = (
    avg_daily["avg_daily_boardings"].round(4)
)

print(avg_daily)

  year_month  avg_daily_boardings
0    2024-01             470.3548
1    2024-02             469.2069
2    2024-03             468.7419
3    2024-04             464.4333
4    2024-05             473.2581
5    2024-06             464.1333


In [9]:
agg_ridership_by_month = agg_ridership_by_month.drop(
    columns=["avg_daily_boardings"]
)

agg_ridership_by_month = agg_ridership_by_month.merge(
    avg_daily,
    on="year_month",
    how="left",
    validate="one_to_one"
)

agg_ridership_by_month["total_revenue"] = (
    agg_ridership_by_month["total_revenue"].round(2)
)

agg_ridership_by_month["avg_daily_boardings"] = (
    agg_ridership_by_month["avg_daily_boardings"].round(4)
)

agg_ridership_by_month = agg_ridership_by_month.sort_values(
    "year_month"
)

print(agg_ridership_by_month)

  year_month  total_boardings  total_revenue  unique_cards  \
0    2024-01            14581       26798.67          9446   
1    2024-02            13607       25176.67          9039   
2    2024-03            14531       26589.69          9428   
3    2024-04            13933       25620.73          9257   
4    2024-05            14671       26714.24          9512   
5    2024-06            13924       25590.24          9154   

   avg_daily_boardings  
0             470.3548  
1             469.2069  
2             468.7419  
3             464.4333  
4             473.2581  
5             464.1333  


In [10]:
output_path = processed_dir / "agg_ridership_by_month.csv"

agg_ridership_by_month.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Rows:", len(agg_ridership_by_month))
print("Columns:", len(agg_ridership_by_month.columns))

Saved: ..\data\processed\agg_ridership_by_month.csv
Rows: 6
Columns: 5


In [11]:
hourly_base = (
    fact.groupby(["hour", "day_type"])
    .agg(
        total_boardings=("passenger_count", "sum"),
        avg_boardings_per_day=("passenger_count", "mean")
    )
    .reset_index()
)

hourly_base["avg_boardings_per_day"] = (
    hourly_base["avg_boardings_per_day"].round(4)
)

print(hourly_base.head(10))

   hour day_type  total_boardings  avg_boardings_per_day
0     0  Weekday              150                 1.0490
1     0  Weekend               48                 1.1163
2     1  Weekday              120                 1.0526
3     1  Weekend               46                 1.0455
4     2  Weekday              160                 1.0811
5     2  Weekend               34                 1.0303
6     3  Weekday              116                 1.0450
7     3  Weekend               69                 1.0781
8     4  Weekday              163                 1.0584
9     4  Weekend               66                 1.1786


In [12]:
hourly_base["daily_total"] = (
    hourly_base.groupby("day_type")["total_boardings"]
    .transform("sum")
)

hourly_base["pct_of_daily_total"] = (
    hourly_base["total_boardings"] / hourly_base["daily_total"]
).round(4)

agg_ridership_by_hour = (
    hourly_base
    .drop(columns="daily_total")
    .sort_values(["day_type", "hour"])
    .reset_index(drop=True)
)

print(agg_ridership_by_hour.head(10))

   hour day_type  total_boardings  avg_boardings_per_day  pct_of_daily_total
0     0  Weekday              150                 1.0490              0.0025
1     1  Weekday              120                 1.0526              0.0020
2     2  Weekday              160                 1.0811              0.0026
3     3  Weekday              116                 1.0450              0.0019
4     4  Weekday              163                 1.0584              0.0027
5     5  Weekday              381                 1.0702              0.0063
6     6  Weekday             1572                 1.0738              0.0258
7     7  Weekday             4823                 1.0730              0.0793
8     8  Weekday             7353                 1.0763              0.1209
9     9  Weekday             5508                 1.0758              0.0905


In [13]:
output_path = processed_dir / "agg_ridership_by_hour.csv"

agg_ridership_by_hour.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Rows:", len(agg_ridership_by_hour))
print("Columns:", len(agg_ridership_by_hour.columns))

Saved: ..\data\processed\agg_ridership_by_hour.csv
Rows: 48
Columns: 5


In [14]:
agg_route_performance = (
    fact.groupby(
        [
            "route_id",
            "route_name",
            "mode",
            "route_district",
            "year_month"
        ]
    )
    .agg(
        total_boardings=("passenger_count", "sum"),
        total_revenue=("revenue", "sum"),
        avg_load_factor=("load_factor", "mean"),
        route_length_km=("route_length_km", "first")
    )
    .reset_index()
)

agg_route_performance = agg_route_performance.rename(
    columns={"route_district": "district"}
)

agg_route_performance["boardings_per_km"] = (
    agg_route_performance["total_boardings"]
    / agg_route_performance["route_length_km"]
).round(4)

agg_route_performance["total_revenue"] = (
    agg_route_performance["total_revenue"].round(2)
)

agg_route_performance["avg_load_factor"] = (
    agg_route_performance["avg_load_factor"].round(4)
)

agg_route_performance["month_start"] = pd.to_datetime(
    agg_route_performance["year_month"] + "-01"
).dt.strftime("%Y-%m-%d")

agg_route_performance = agg_route_performance[
    [
        "route_id",
        "route_name",
        "mode",
        "district",
        "year_month",
        "total_boardings",
        "total_revenue",
        "boardings_per_km",
        "avg_load_factor",
        "month_start",
        "route_length_km"
    ]
].sort_values(
    ["year_month", "total_boardings"],
    ascending=[True, False]
).reset_index(drop=True)

print(agg_route_performance.head())
print("Rows:", len(agg_route_performance))
print("Columns:", len(agg_route_performance.columns))

  route_id             route_name   mode district year_month  total_boardings  \
0     R032        Thomas Dam Line  Metro   Centro    2024-01              292   
1     R051       Daniel Lake Line    Bus   Centro    2024-01              285   
2     R039  Carlson Mountain Line    Bus   Centro    2024-01              280   
3     R031     Wilkerson Row Line   Tram   Centro    2024-01              278   
4     R021      Martin Knoll Line   Tram  Westend    2024-01              277   

   total_revenue  boardings_per_km  avg_load_factor month_start  \
0         498.01           12.4255           0.0019  2024-01-01   
1         521.42           58.1633           0.0142  2024-01-01   
2         501.23           66.6667           0.0142  2024-01-01   
3         516.28           16.4497           0.0055  2024-01-01   
4         526.82           11.7373           0.0057  2024-01-01   

   route_length_km  
0             23.5  
1              4.9  
2              4.2  
3             16.9  
4    

In [15]:
key_columns = [
    "route_id",
    "route_name",
    "mode",
    "district",
    "year_month"
]

assert not agg_route_performance[key_columns].isna().any().any(), (
    "Null values found in key columns!"
)

print("Key-column null check passed.")

Key-column null check passed.


In [16]:
output_path = processed_dir / "agg_route_performance.csv"

agg_route_performance.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Rows:", len(agg_route_performance))
print("Columns:", len(agg_route_performance.columns))

Saved: ..\data\processed\agg_route_performance.csv
Rows: 360
Columns: 11


In [17]:
agg_mode_summary = (
    fact.groupby(["mode", "year_month"])
    .agg(
        total_boardings=("passenger_count", "sum"),
        total_revenue=("revenue", "sum"),
        route_count=("route_id", "nunique")
    )
    .reset_index()
)

agg_mode_summary["boardings_per_route"] = (
    agg_mode_summary["total_boardings"]
    / agg_mode_summary["route_count"]
).round(4)

agg_mode_summary["total_revenue"] = (
    agg_mode_summary["total_revenue"].round(2)
)

agg_mode_summary = agg_mode_summary.sort_values(["mode", "year_month"])

agg_mode_summary["boardings_mom_growth"] = (
    agg_mode_summary
    .groupby("mode")["total_boardings"]
    .pct_change()
    .round(4)
)

agg_mode_summary["month_start"] = pd.to_datetime(
    agg_mode_summary["year_month"] + "-01"
).dt.strftime("%Y-%m-%d")

agg_mode_summary = (
    agg_mode_summary
    [
        [
            "mode",
            "year_month",
            "total_boardings",
            "total_revenue",
            "route_count",
            "boardings_per_route",
            "boardings_mom_growth",
            "month_start"
        ]
    ]
    .sort_values(["year_month", "total_boardings"], ascending=[True, False])
    .reset_index(drop=True)
)

print(agg_mode_summary)
print("Rows:", len(agg_mode_summary))

     mode year_month  total_boardings  total_revenue  route_count  \
0     Bus    2024-01             8332       15363.63           34   
1    Tram    2024-01             4809        8823.31           20   
2   Metro    2024-01             1440        2611.73            6   
3     Bus    2024-02             7676       14213.12           34   
4    Tram    2024-02             4668        8575.81           20   
5   Metro    2024-02             1263        2387.74            6   
6     Bus    2024-03             8167       14929.71           34   
7    Tram    2024-03             4932        9098.29           20   
8   Metro    2024-03             1432        2561.69            6   
9     Bus    2024-04             7934       14583.72           34   
10   Tram    2024-04             4658        8570.28           20   
11  Metro    2024-04             1341        2466.73            6   
12    Bus    2024-05             8373       15283.89           34   
13   Tram    2024-05             4

In [18]:
key_columns = ["mode", "year_month"]

assert not agg_mode_summary[key_columns].isna().any().any(), (
    "Null values found in key columns!"
)

print("Key-column null check passed.")

Key-column null check passed.


In [19]:
output_path = processed_dir / "agg_mode_summary.csv"

agg_mode_summary.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Rows:", len(agg_mode_summary))
print("Columns:", len(agg_mode_summary.columns))

Saved: ..\data\processed\agg_mode_summary.csv
Rows: 18
Columns: 8


In [20]:
agg_fare_type_mix = (
    fact.groupby(["fare_type", "year_month"])
    .agg(
        boardings=("passenger_count", "sum"),
        revenue=("revenue", "sum")
    )
    .reset_index()
)

agg_fare_type_mix["monthly_total"] = (
    agg_fare_type_mix
    .groupby("year_month")["boardings"]
    .transform("sum")
)

agg_fare_type_mix["pct_of_monthly_boardings"] = (
    agg_fare_type_mix["boardings"]
    / agg_fare_type_mix["monthly_total"]
).round(4)

agg_fare_type_mix["revenue"] = (
    agg_fare_type_mix["revenue"].round(2)
)

agg_fare_type_mix["month_start"] = pd.to_datetime(
    agg_fare_type_mix["year_month"] + "-01"
).dt.strftime("%Y-%m-%d")

agg_fare_type_mix = (
    agg_fare_type_mix
    .drop(columns="monthly_total")
    .sort_values(["year_month", "boardings"], ascending=[True, False])
    .reset_index(drop=True)
)

print(agg_fare_type_mix)
print("Rows:", len(agg_fare_type_mix))

      fare_type year_month  boardings   revenue  pct_of_monthly_boardings  \
0         Adult    2024-01       8072  20173.80                    0.5536   
1       Student    2024-01       2218   3330.15                    0.1521   
2        Senior    2024-01       1738   2077.91                    0.1192   
3   Pass Holder    2024-01       1416     83.28                    0.0971   
4         Child    2024-01       1137   1133.53                    0.0780   
5         Adult    2024-02       7635  19100.71                    0.5611   
6       Student    2024-02       2036   3048.67                    0.1496   
7        Senior    2024-02       1591   1903.05                    0.1169   
8   Pass Holder    2024-02       1294     72.38                    0.0951   
9         Child    2024-02       1051   1051.86                    0.0772   
10        Adult    2024-03       8005  20006.92                    0.5509   
11      Student    2024-03       2192   3277.76                    0.1508   

In [21]:
key_columns = ["fare_type", "year_month"]

assert not agg_fare_type_mix[key_columns].isna().any().any(), (
    "Null values found in key columns!"
)

print("Key-column null check passed.")

Key-column null check passed.


In [22]:
output_path = processed_dir / "agg_fare_type_mix.csv"

agg_fare_type_mix.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Rows:", len(agg_fare_type_mix))
print("Columns:", len(agg_fare_type_mix.columns))

Saved: ..\data\processed\agg_fare_type_mix.csv
Rows: 30
Columns: 6


In [23]:
agg_stop_activity = (
    fact.groupby(
        ["stop_id", "stop_name", "stop_district", "zone", "has_shelter"]
    )
    .agg(
        total_boardings=("passenger_count", "sum")
    )
    .reset_index()
)

agg_stop_activity = agg_stop_activity.rename(
    columns={"stop_district": "district"}
)

agg_stop_activity["rank_within_district"] = (
    agg_stop_activity
    .groupby("district")["total_boardings"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

agg_stop_activity["has_shelter"] = (
    agg_stop_activity["has_shelter"].astype(int)
)

agg_stop_activity = (
    agg_stop_activity[
        [
            "stop_id",
            "stop_name",
            "district",
            "zone",
            "total_boardings",
            "rank_within_district",
            "has_shelter"
        ]
    ]
    .sort_values(
        ["district", "rank_within_district", "stop_id"]
    )
    .reset_index(drop=True)
)

print(agg_stop_activity.head(10))
print("Rows:", len(agg_stop_activity))

  stop_id             stop_name district    zone  total_boardings  \
0   S0047     Joseph Coves Stop   Centro  Zone 3              237   
1   S0314      Davis Locks Stop   Centro  Zone 2              233   
2   S0394     Tucker Mills Stop   Centro  Zone 2              229   
3   S0103  Jennifer Lights Stop   Centro  Zone 1              228   
4   S0323     Brandon Camp Stop   Centro  Zone 1              228   
5   S0021      Lewis Locks Stop   Centro  Zone 3              226   
6   S0037      Emily Hills Stop   Centro  Zone 1              226   
7   S0391    Nicole Drives Stop   Centro  Zone 1              226   
8   S0215    Deborah Grove Stop   Centro  Zone 3              225   
9   S0014      James Plain Stop   Centro  Zone 3              221   

   rank_within_district  has_shelter  
0                     1            0  
1                     2            1  
2                     3            0  
3                     4            0  
4                     4            0  
5     

In [24]:
key_columns = [
    "stop_id",
    "stop_name",
    "district",
    "zone"
]

assert not agg_stop_activity[key_columns].isna().any().any(), (
    "Null values found in key columns!"
)

print("Key-column null check passed.")

Key-column null check passed.


In [25]:
output_path = processed_dir / "agg_stop_activity.csv"

agg_stop_activity.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Rows:", len(agg_stop_activity))
print("Columns:", len(agg_stop_activity.columns))

Saved: ..\data\processed\agg_stop_activity.csv
Rows: 400
Columns: 7


In [26]:
agg_vehicle_utilisation = (
    fact.groupby(
        ["vehicle_id", "vehicle_type", "depot", "vehicle_age_band"]
    )
    .agg(
        total_boardings=("passenger_count", "sum"),
        avg_load_factor=("load_factor", "mean"),
        active_days=("date", "nunique"),
        capacity=("capacity", "first")
    )
    .reset_index()
)

agg_vehicle_utilisation["avg_load_factor"] = (
    agg_vehicle_utilisation["avg_load_factor"].round(4)
)

agg_vehicle_utilisation = (
    agg_vehicle_utilisation
    .sort_values("total_boardings", ascending=False)
    .reset_index(drop=True)
)

print(agg_vehicle_utilisation.head())
print("Rows:", len(agg_vehicle_utilisation))
print("Columns:", len(agg_vehicle_utilisation.columns))

  vehicle_id vehicle_type            depot vehicle_age_band  total_boardings  \
0      V0176         Tram      North Depot              20+              852   
1      V0116         Tram  Riverside Depot            11-20              814   
2      V0053         Tram    Central Depot              20+              801   
3      V0111         Tram      North Depot              0-5              788   
4      V0105         Tram      North Depot              20+              783   

   avg_load_factor  active_days  capacity  
0           0.0061          179       179  
1           0.0059          180       188  
2           0.0062          178       177  
3           0.0049          179       227  
4           0.0061          179       182  
Rows: 180
Columns: 8


In [27]:
key_columns = [
    "vehicle_id",
    "vehicle_type",
    "depot",
    "vehicle_age_band"
]

assert not agg_vehicle_utilisation[key_columns].isna().any().any(), (
    "Null values found in key columns!"
)

print("Key-column null check passed.")

Key-column null check passed.


In [28]:
output_path = processed_dir / "agg_vehicle_utilisation.csv"

agg_vehicle_utilisation.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Rows:", len(agg_vehicle_utilisation))
print("Columns:", len(agg_vehicle_utilisation.columns))

Saved: ..\data\processed\agg_vehicle_utilisation.csv
Rows: 180
Columns: 8


In [29]:
PEAK_PERIOD_ORDER = {
    "Night (00-03)": 1,
    "Early (04-06)": 2,
    "AM Peak (07-09)": 3,
    "Midday (10-15)": 4,
    "PM Peak (16-18)": 5,
    "Evening (19-23)": 6,
}

agg_peak_analysis = (
    fact.groupby(["time_period", "mode"])
    .agg(
        total_boardings=("passenger_count", "sum"),
        avg_load_factor=("load_factor", "mean")
    )
    .reset_index()
)

agg_peak_analysis["mode_total"] = (
    agg_peak_analysis
    .groupby("mode")["total_boardings"]
    .transform("sum")
)

agg_peak_analysis["pct_of_mode_total"] = (
    agg_peak_analysis["total_boardings"]
    / agg_peak_analysis["mode_total"]
).round(4)

agg_peak_analysis["avg_load_factor"] = (
    agg_peak_analysis["avg_load_factor"].round(4)
)

agg_peak_analysis["period_order"] = (
    agg_peak_analysis["time_period"].map(PEAK_PERIOD_ORDER)
)

agg_peak_analysis = (
    agg_peak_analysis
    .drop(columns="mode_total")
    .sort_values(["mode", "total_boardings"], ascending=[True, False])
    .reset_index(drop=True)
)

print(agg_peak_analysis)
print("Rows:", len(agg_peak_analysis))
print("Columns:", len(agg_peak_analysis.columns))

        time_period   mode  total_boardings  avg_load_factor  \
0   PM Peak (16-18)    Bus            15341           0.0135   
1   AM Peak (07-09)    Bus            14041           0.0135   
2    Midday (10-15)    Bus            11950           0.0136   
3   Evening (19-23)    Bus             4806           0.0135   
4     Early (04-06)    Bus             1650           0.0134   
5     Night (00-03)    Bus              423           0.0126   
6   PM Peak (16-18)  Metro             2620           0.0019   
7   AM Peak (07-09)  Metro             2457           0.0019   
8    Midday (10-15)  Metro             2085           0.0018   
9   Evening (19-23)  Metro              828           0.0018   
10    Early (04-06)  Metro              265           0.0018   
11    Night (00-03)  Metro               75           0.0019   
12  PM Peak (16-18)   Tram             9086           0.0055   
13  AM Peak (07-09)   Tram             8342           0.0054   
14   Midday (10-15)   Tram             7

In [30]:
key_columns = ["time_period", "mode"]

assert not agg_peak_analysis[key_columns].isna().any().any(), (
    "Null values found in key columns!"
)

print("Key-column null check passed.")

Key-column null check passed.


In [31]:

output_path = processed_dir / "agg_peak_analysis.csv"

agg_peak_analysis.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Rows:", len(agg_peak_analysis))
print("Columns:", len(agg_peak_analysis.columns))

Saved: ..\data\processed\agg_peak_analysis.csv
Rows: 18
Columns: 6


In [32]:
agg_transfer_behaviour = (
    fact.groupby(["year_month", "mode"])
    .agg(
        transfer_count=("transfer_flag", "sum"),
        validations=("transfer_flag", "count")
    )
    .reset_index()
)

agg_transfer_behaviour["transfer_rate"] = (
    agg_transfer_behaviour["transfer_count"]
    / agg_transfer_behaviour["validations"]
).round(4)

agg_transfer_behaviour["month_start"] = pd.to_datetime(
    agg_transfer_behaviour["year_month"] + "-01"
).dt.strftime("%Y-%m-%d")

agg_transfer_behaviour = (
    agg_transfer_behaviour
    .sort_values(["year_month", "mode"])
    .reset_index(drop=True)
)

print(agg_transfer_behaviour)
print("Rows:", len(agg_transfer_behaviour))
print("Columns:", len(agg_transfer_behaviour.columns))

   year_month   mode  transfer_count  validations  transfer_rate month_start
0     2024-01    Bus            2903         7721          0.376  2024-01-01
1     2024-01  Metro             499         1331         0.3749  2024-01-01
2     2024-01   Tram            1735         4508         0.3849  2024-01-01
3     2024-02    Bus            2642         7141           0.37  2024-02-01
4     2024-02  Metro             413         1180           0.35  2024-02-01
5     2024-02   Tram            1666         4286         0.3887  2024-02-01
6     2024-03    Bus            2923         7560         0.3866  2024-03-01
7     2024-03  Metro             500         1312         0.3811  2024-03-01
8     2024-03   Tram            1747         4589         0.3807  2024-03-01
9     2024-04    Bus            2827         7389         0.3826  2024-04-01
10    2024-04  Metro             435         1241         0.3505  2024-04-01
11    2024-04   Tram            1701         4363         0.3899  2024-04-01

In [33]:
key_columns = ["year_month", "mode"]

assert not agg_transfer_behaviour[key_columns].isna().any().any(), (
    "Null values found in key columns!"
)

print("Key-column null check passed.")

Key-column null check passed.


In [34]:
output_path = processed_dir / "agg_transfer_behaviour.csv"

agg_transfer_behaviour.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Rows:", len(agg_transfer_behaviour))
print("Columns:", len(agg_transfer_behaviour.columns))

Saved: ..\data\processed\agg_transfer_behaviour.csv
Rows: 18
Columns: 6


In [35]:
from datetime import datetime

output_names = [
    "agg_ridership_by_month",
    "agg_ridership_by_hour",
    "agg_route_performance",
    "agg_mode_summary",
    "agg_fare_type_mix",
    "agg_stop_activity",
    "agg_vehicle_utilisation",
    "agg_peak_analysis",
    "agg_transfer_behaviour"
]

manifest_rows = []

for name in output_names:
    file_path = processed_dir / f"{name}.csv"

    df = pd.read_csv(file_path)

    manifest_rows.append({
        "file_name": file_path.name,
        "row_count": len(df),
        "column_count": len(df.columns),
        "generated_at": datetime.now().isoformat(timespec="seconds")
    })

manifest = pd.DataFrame(manifest_rows)

manifest_path = processed_dir / "_manifest.csv"

manifest.to_csv(
    manifest_path,
    index=False,
    encoding="utf-8"
)

print(manifest)
print()
print("Manifest saved:", manifest_path)

                     file_name  row_count  column_count         generated_at
0   agg_ridership_by_month.csv          6             5  2026-11-22T18:18:59
1    agg_ridership_by_hour.csv         48             5  2026-11-22T18:18:59
2    agg_route_performance.csv        360            11  2026-11-22T18:18:59
3         agg_mode_summary.csv         18             8  2026-11-22T18:18:59
4        agg_fare_type_mix.csv         30             6  2026-11-22T18:18:59
5        agg_stop_activity.csv        400             7  2026-11-22T18:18:59
6  agg_vehicle_utilisation.csv        180             8  2026-11-22T18:18:59
7        agg_peak_analysis.csv         18             6  2026-11-22T18:18:59
8   agg_transfer_behaviour.csv         18             6  2026-11-22T18:18:59

Manifest saved: ..\data\processed\_manifest.csv


In [36]:
from pathlib import Path

processed_dir = Path("../data/processed")

expected_files = [
    "agg_ridership_by_month.csv",
    "agg_ridership_by_hour.csv",
    "agg_route_performance.csv",
    "agg_mode_summary.csv",
    "agg_fare_type_mix.csv",
    "agg_stop_activity.csv",
    "agg_vehicle_utilisation.csv",
    "agg_peak_analysis.csv",
    "agg_transfer_behaviour.csv",
    "_manifest.csv"
]

missing_files = [
    name for name in expected_files
    if not (processed_dir / name).exists()
]

assert not missing_files, f"Missing files: {missing_files}"

print("Final export check passed.")
print(f"All {len(expected_files)} files exist in:", processed_dir)

Final export check passed.
All 10 files exist in: ..\data\processed
